<hr>

# 🧫 DATA CLEANING 


<style>
h1 {
    text-align: left;
    color: blue;
    font-weight: bold;
}

</style>
<hr>

<style>
h3 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

### 📂 IMPORTs

In [ ]:
# import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import seaborn as sns

plt.style.use('ggplot')
pd.set_option('display.max_columns', 200) # to display all columns in the dataframe

<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### DISPLAY_DATA FUNCTION 

In [ ]:
import pandas as pd
# define function display_data to explore each dataset
def display_data(df: pd.DataFrame):
    # ----------------------------------------
    # Display shape and head of the dataframe
    # ----------------------------------------
    print(f"{df.name} Dataset Shape: {df.shape[0]} rows and {df.shape[1]} columns\n")
    display(df.head(3))
    print(100*"-" + "\n")

    # ----------------------------------------------------------
    # Display data types & missing values of each column in df
    # ----------------------------------------------------------
    print("Data Types & Missing Values of Each Column:")
    display(df.info())

    # ---------------------------------------------------
    # Display Info & unique values for each column in df    
    # ---------------------------------------------------
    for col in df.columns:
        unique_vals = df[col].unique()
        print(f"Column: {col}")
        print(f"Unique values ({len(unique_vals)}): {unique_vals}\n")

    print(f"\nData types check:")
    for col in df.columns:
        dtype = df[col].dtype
        unique_vals = df[col].nunique()
        print(f"  ➡️{col:<25} {str(dtype):<10} [{unique_vals}] unique values")

<hr>

## 0 - DATA CLEANING OVERVIEW


<style>
h1 {
    text-align: center;
    color: black;
    font-weight: bold;
}
</style>

<style>
h2 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h3 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h4 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>
<hr>

Note observations and changes to be made on the data:
- note a
- note b
- note c
- change 1
- change 2
- change 3

<hr>

## 1 - STANDARDIZE NAMES


<style>
h1 {
    text-align: center;
    color: black;
    font-weight: bold;
}
</style>

<style>
h2 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h3 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h4 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>
<hr>

<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### ➕ COMBINE - 6 TXT FILES INTO 1 CSV
Dropped unnecessary columns

In [ ]:
import pandas as pd

# List of DVF files
dvf_files = [
    "../data/raw/ValeursFoncieres-2020-S2.txt",
    "../data/raw/ValeursFoncieres-2021.txt",
    "../data/raw/ValeursFoncieres-2022.txt",
    "../data/raw/ValeursFoncieres-2023.txt",
    "../data/raw/ValeursFoncieres-2024.txt",
    "../data/raw/ValeursFoncieres-2025-S1.txt"
]

# Columns to keep
usecols = [
    "Date mutation",
    "Nature mutation",
    "Valeur fonciere",
    "Commune",
    "Type local",
    "Surface reelle bati",
    "Nombre pieces principales",
    "Nombre de lots",
    "Surface terrain", 
    "No voie",    # street number
    "B/T/Q",      # building / type / quartier
    "Voie",       # street name
    "Code postal" # postal code
]

# Output CSV
output_file = "../data/processed/RAW_ValeursFoncieres_2020-S2_2025-S1.csv"

# Fix mixed types
dtype_dict = {
    "Code postal": str,
    "Voie": str,
    "No voie": str,
    "B/T/Q": str,
    "Type local": str,
    "department_code": str,
    "surface_type": str

}

with open(output_file, "w", newline="", encoding="utf-8") as f_out:
    header_written = False

    for file in dvf_files:
        print(f"Processing: {file}")

        for chunk in pd.read_csv(
            file,
            sep="|",
            usecols=usecols,
            chunksize=50_000,
            encoding="latin-1",
            decimal=",",
            low_memory=False,
            dtype=dtype_dict
        ):
            # -------------------------
            # FIX ENCODING
            # -------------------------
            chunk["Type local"] = (
                chunk["Type local"]
                .astype(str)
                .str.encode("latin-1", errors="ignore")
                .str.decode("utf-8", errors="ignore")
                .str.strip()
            )

            # -------------------------
            # CONVERT NUMERIC
            # -------------------------
            chunk["Valeur fonciere"] = pd.to_numeric(chunk["Valeur fonciere"], errors="coerce")
            chunk["Surface reelle bati"] = pd.to_numeric(chunk["Surface reelle bati"], errors="coerce")
            chunk["Surface terrain"] = pd.to_numeric(chunk["Surface terrain"], errors="coerce")

            # Drop rows with missing price
            chunk = chunk.dropna(subset=["Valeur fonciere"])

            # -------------------------
            # PRICE PER M² (SMART FALLBACK)
            # -------------------------
            chunk["effective_surface"] = chunk["Surface reelle bati"]

            # If built surface is missing or zero → use terrain
            mask = chunk["effective_surface"].isna() | (chunk["effective_surface"] == 0)
            chunk.loc[mask, "effective_surface"] = chunk.loc[mask, "Surface terrain"]

            # Compute price per m²
            chunk["value_per_m2"] = chunk["Valeur fonciere"] / chunk["effective_surface"]

            # Clean invalid values
            chunk.loc[
                chunk["effective_surface"].isna() | (chunk["effective_surface"] == 0),
                "value_per_m2"
            ] = None

            # Optional: track which surface was used
            chunk["surface_type"] = "bati"
            chunk.loc[mask, "surface_type"] = "terrain"

            # -------------------------
            # EXTRACT DEPARTEMENT
            # -------------------------
            chunk["department_code"] = chunk["Code postal"].astype(str).str[:2]

            # -------------------------
            # WRITE TO CSV
            # -------------------------
            chunk.to_csv(
                f_out,
                index=False,
                header=not header_written
            )

            header_written = True
            del chunk  # free memory

print(f"✅ TXT Files aggregated & converted into one CSV File")
print(f"✅ CSV File saved at {output_file}")

### ⭐ **Valeurs Foncieres 2020-2025**
Standardize column names

| Original Column Name      |      Standardized      | Notes / Description                                           |
| ------------------------- | ---------------------- | ------------------------------------------------------------ |
| **Date mutation**             | `transaction_date`       | Date when the property was sold                        |
| **Nature mutation**           | `transaction_type`       | Nature of transaction (sale, donation, etc.)                 |
| **Valeur fonciere**           | `property_value`         | Transaction price in euros                                   |
| **No voie**                   | `street_number`          | Street number of the property                                |
| **B/T/Q**                     | `btq_code`               | Bâtiment / Type / Quartier code for identification           |
| **Voie**                      | `street_name`            | Full street name                                             |
| **Code postal**               | `postal_code`            | Postal code (categorical, not numeric)                       |
| **Commune**                   | `commune_name`                | Commune name                                                 |
| **Nombre de lots**            | `number_of_lots`         | Number of lots in the property                               |
| **Type local**                | `property_type`          | Property type (Apartment, House, etc.)                       |
| **Surface reelle bati**       | `building_real_surface`  | Built area in square meters                                  |
| **Nombre pieces principales** | `main_room_count`        | Number of main rooms (living / bedrooms)                     |
| **Surface terrain**           | `land_surface`           | Land area in square meters                                   |
| **effective_surface**              | `effective_surface`      | Surface used to calculate price/m² (building or land)        |
| **value_per_m2**                   | `value_per_m2`           | Value per m² in euros                                        |
| **surface_type**              | `surface_type`           | Indicates if effective_surface comes from a building or land |
| **departement**               | `department_code`        | 2-digit INSEE department code                                |


<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### DISPLAY HEAD - RAW Valeurs Foncieres

In [ ]:
import pandas as pd
df = pd.read_csv("../data/processed/RAW_ValeursFoncieres_2020-S2_2025-S1.csv")

# Display df DataFrame
print("RAW Valeur Foncieres:")
display(df.head())
print("Dataset Shape:", df.shape[0], "rows and", df.shape[1], "columns\n")

<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### DISPLAY UNIQUE VALUES - RAW Valeurs Foncieres

In [ ]:
# Display data types & missing values of each column in df
print("Data Types & Missing Values of Each Column:")
display(df.info())

# Loop through each column and print unique values
for col in df.columns:
    unique_vals = df[col].unique()
    print(f"Column: {col}")
    print(f"Unique values ({len(unique_vals)}): {unique_vals}\n")

print(f"\nData types check:")
for col in df.columns:
    dtype = df[col].dtype
    unique_vals = df[col].nunique()
    print(f"  ➡️{col:<25} {str(dtype):<10} [{unique_vals}] unique values")

<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### ✏️ RENAME - COLUMNS

In [ ]:
import pandas as pd

# -------------------
# Load main dataset
# -------------------
df = pd.read_csv("../data/processed/RAW_ValeursFoncieres_2020-S2_2025-S1.csv")

# ---------------------------------------------
# Define the Renaming Dictionary
# ---------------------------------------------
rename_columns = {
    "Date mutation": "transaction_date",
    "Nature mutation": "transaction_type",
    "Valeur fonciere": "property_value",
    "No voie": "street_number",
    "B/T/Q": "btq_code",
    "Voie": "street_name",
    "Code postal": "postal_code",
    "Commune": "commune",
    "Nombre de lots": "number_of_lots",
    "Type local": "property_type",
    "Surface reelle bati": "building_real_surface",
    "Nombre pieces principales": "main_room_count",
    "Surface terrain": "land_surface",
    "surface_used": "effective_surface",
    "Prix_m2": "value_per_m2",
    "surface_type": "surface_type",
    "departement": "department_code"
}

# ---------------------------------------------
# Rename columns using the rename Dictionary
# ---------------------------------------------
df.rename(columns=rename_columns, inplace=True)

# --------------------------------
# Save dataset in a new CSV file
# --------------------------------
df.to_csv("../data/processed/INTERIM_01_standard_ValeursFoncieres.csv", index=False)

print(f"✅ Dataset standardized and saved at ../data/processed/INTERIM_01_standard_ValeursFoncieres.csv")

In [ ]:
### ⭐ **Valeurs Foncieres 2020-2025**
Standardize column names

| Original Column Name      |      Standardized      | Notes / Description                                           |
| ------------------------- | ---------------------- | ------------------------------------------------------------ |
| Date mutation             | transaction_date       | Date when the property was sold                              |
| Nature mutation           | transaction_type       | Nature of transaction (sale, donation, etc.)                 |
| Valeur fonciere           | property_value         | Transaction price in euros                                   |
| No voie                   | street_number          | Street number of the property                                |
| B/T/Q                     | btq_code               | Bâtiment / Type / Quartier code for identification           |
| Voie                      | street_name            | Full street name                                             |
| Code postal               | postal_code            | Postal code (categorical, not numeric)                       |
| Commune                   | commune                | Commune name                                                 |
| Nombre de lots            | number_of_lots         | Number of lots in the property                               |
| Type local                | property_type          | Property type (Apartment, House, etc.)                       |
| Surface reelle bati       | building_real_surface  | Built area in square meters                                  |
| Nombre pieces principales | main_room_count        | Number of main rooms (living / bedrooms)                     |
| Surface terrain           | land_surface           | Land area in square meters                                   |
| surface_used              | effective_surface      | Surface used to calculate price/m² (building or land)        |
| Prix_m2                   | value_per_m2           | Value per m² in euros                                        |
| surface_type              | surface_type           | Indicates if effective_surface comes from a building or land |
| departement               | department_code        | 2-digit INSEE department code                                |


<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### 🟨 DISPLAY - Standard ValeursFoncieres

In [ ]:
# Display Standardized DataFrame
print("Standardized DataFrame:")
print(df.shape[0], "rows and", df.shape[1], "columns")
display(df.head())

# Display standardized column names
print("Standardized Column Names:")
display(df.info())

<hr>

## 2 - INVALID VALUES


<style>
h1 {
    text-align: center;
    color: black;
    font-weight: bold;
}
</style>

<style>
h2 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h3 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h4 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>
<hr>

<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### LOAD & DISPLAY STANDARDIZED - Valeurs Foncieres

In [ ]:
import pandas as pd
df = pd.read_csv("../data/processed/INTERIM_01_standard_ValeursFoncieres.csv")
print("Standardized Valeurs Foncieres Dataset Shape:", df.shape[0], "rows and", df.shape[1], "columns")
display(df.head())

<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### DISPLAY UNIQUE VALUES - Valeurs Foncieres

In [ ]:
# Loop through each column and print unique values
print(100 * "-")
for col in df.columns:
    unique_vals = df[col].unique()
    print(f"➡️ {col} ({len(unique_vals)}) 🧾 unique values:")
    print(f"{unique_vals}")
    print(100 * "-")

In [ ]:
df_communes = pd.read_csv("../data/raw/communes-france-2025.csv")
df_communes.head()

<style>
h3 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

### DEAL with INVALID VALUES - Valeurs Foncieres

In [ ]:
# ----------------------------------------
# Column: transaction_type
# ----------------------------------------
# fix transaction_type encoding issues to ensure consistent values
# replace values to english
df['transaction_type'] = df['transaction_type'].replace({
        "Vente": "Sale",
        "Vente en l'Ã©tat futur d'achÃ¨vement": "Sale in future state of completion",
        "Vente terrain Ã\xa0 bÃ¢tir": "Sale of unbuilt land",
        "Echange": "Exchange",
        "Adjudication": "Auction",
        "Expropriation": "Expropriation"})

# --------------------
# Column: area_type
# --------------------
# area_type: replace 'bati' with 'built' and 'terrain' with 'land'
df['area_type'] = df['area_type'].replace({
    "bati": "built",    
    "terrain": "land"})


# ------------------------
# Column: building_code
# ------------------------
# fix building_code inconsistencies: replace invalid values with NaN and standardize 'b' to 'B'
invalid_values = ['-', '.', '*']
df["building_code"] = df["building_code"].replace(invalid_values, pd.NA)
df['building_code'] = df['building_code'].replace({ 'b': 'B'})


# --------------------------
# Column: building_area_m2
# --------------------------
# because it is probably not a built property if building_area_m2 is NaN
# as building area is not relevant for land sales
# fill NaN with 0 
df["building_area_m2"] = df["building_area_m2"].fillna(0)

# --------------------------
# Column: main_rooms
# --------------------------
# because it is probably not a built property if main_rooms is NaN
# fill NaN with 0 
df["main_rooms"] = df["main_rooms"].fillna(0)

# --------------------------
# Column: street_number
# --------------------------
# nulls in street_number are common and not necessarily invalid, so we will keep them as is for now


# --------------------------------------------------------------------------------------
# Column: commune
# --------------------------------------------------------------------------------------

# --- REPLACE COMMUNE NAMES in DataFrame with OFFICIAL NAMES FROM COMMUNES DATASET TO ENSURE CONSISTENCY ---
# load communes dataset with INSEE codes and official names
df_communes = pd.read_csv("../data/processed/CLEAN_communes-france-2025.csv")

# create a mapping dictionary from the communes dataset
commune_mapping = dict(zip(df_communes["Nom officiel de la commune"], df_communes["Nom officiel de la commune"]))

# replace commune names in the main dataset using the mapping dictionary
df["commune"] = df["commune"].replace(commune_mapping)


# --------------------------------------------------------------------------------------------
# SAVE - InvalidValues ValeursFoncieres
# --------------------------------------------------------------------------------------------
df.to_csv("../data/processed/INTERIM_02_InvalidValues_ValeursFoncieres.csv", index=False)
print(f"✅ Dataset with invalid values handled and saved at ../data/processed/INTERIM_02_InvalidValues_ValeursFoncieres.csv")

<hr>

## 3 - DATA TYPES


<style>
h1 {
    text-align: center;
    color: black;
    font-weight: bold;
}
</style>

<style>
h2 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h3 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h4 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>
<hr>

<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### DISPLAY TYPES - DVF 2020-2025

In [ ]:
# Load the dataset
df = pd.read_csv("../data/processed/INTERIM_02_InvalidValues_ValeursFoncieres.csv")

# print data types and unique values count for each column in df
print(f"\nData types check:")
for col in df.columns:
    dtype = df[col].dtype
    unique_vals = df[col].nunique()
    print(f"  ➡️{col:<25} {str(dtype):<10} [{unique_vals}] unique values")

<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### CONVERT - DVF 2020-2025

| Field Name                                 | Python data type             |
|--------------------------------------------|------------------------------|
| **transaction_date**                          | `object` to `date`           |
| **street_number**                                | `float64` to `int64`         |
| **postal_code**                            | `float64` to `object`        |
| **main_rooms**              | `float64` to `int64`         |


In [ ]:
# convert df columns types as follows:
df["transaction_date"] = pd.to_datetime(df["transaction_date"], errors='coerce')
df["street_number"] = df["street_number"].astype('int64')
df["postal_code"] = df["postal_code"].astype('object')
df["main_rooms"] = df["main_rooms"].astype('int64')

# Display Converted column names
print("Converted column names:")
display(df.info())

# Save dataset in a new CSV file
df.to_csv("../data/processed/INTERIM_03_DataType_ValeursFoncieres.csv", index=False)
print(f"✅ Dataset converted to appropriate data types and saved at ../data/processed/INTERIM_03_DataType_ValeursFoncieres.csv")

<hr>

## 4 - NULL VALUES


<style>
h1 {
    text-align: center;
    color: black;
    font-weight: bold;
}
</style>

<style>
h2 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h3 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h4 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>
<hr>

<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### DISPLAY NULLs - Valeurs Foncieres

In [ ]:
import pandas as pd

# Load dataset with converted data types
df = pd.read_csv("../data/processed/INTERIM_03_DataType_ValeursFoncieres.csv")

# Loop through each column and print the number of nulls in each column
for col in df.columns:
    null_count = df[col].isnull().sum()
    print(f"[{col}]: [{null_count}] null values")

print(100*"-")
# display data types and missing values of each column in df
df.info()

<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### DEAL with NULLs - Valeur Foncieres

In [ ]:
# ----------------------------------------
# Column: sale_price_eur
# ----------------------------------------
# drop nulls in sale_price_eur column as it's a critical column for analysis
df = df.dropna(subset=["sale_price_eur"])

# ----------------------------------------
# Column: building_area_m2
# ----------------------------------------
# Fill NaN with 0
df["building_area_m2"] = df["building_area_m2"].fillna(0)

# ----------------------------------------
# Column: property_type
# ----------------------------------------
# Fill NaN with "Unknown" to keep all rows for analysis, 
# as property_type is important for analysis and we don't want to drop rows with missing property_type
df["property_type"] = df["property_type"].fillna("Unknown")

# ----------------------------------------
# Column: main_rooms
# ----------------------------------------
# main_rooms is only relevant for houses/apartments
# fill NaN with 0 if property_type is terrain
df["main_rooms"] = df.apply(lambda row: 0 if pd.isna(row["main_rooms"]) and row["property_type"] == "Terrain" else row["main_rooms"], axis=1)

# ----------------------------------------
# Column: 
# ----------------------------------------
# Fill NaN with 0

# ----------------------------------------
# Save dataset in a new CSV file
# ----------------------------------------
df.to_csv("../data/processed/INTERIM_04_Nulls_ValeursFoncieres.csv", index=False)
print(f"✅ Dataset with null values handled and saved at ../data/processed/INTERIM_04_Nulls_ValeursFoncieres.csv")

<hr>

## 5 - DUPLICATES


<style>
h1 {
    text-align: center;
    color: black;
    font-weight: bold;
}
</style>

<style>
h2 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h3 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h4 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>
<hr>

<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### DISPLAY DUPLICATE ROWS - Valeurs Foncieres

In [ ]:
import pandas as pd

# load dataset with nulls handled
df = pd.read_csv("../data/processed/INTERIM_04_Nulls_ValeursFoncieres.csv")

# Check for duplicate rows in the DataFrame
duplicate_rows = df[df.duplicated()]

# Print the number of duplicate rows
print(f"Number of duplicate rows: {duplicate_rows.shape[0]}")

# Display the duplicate rows if any exist
if not duplicate_rows.empty:
    print("\nDuplicate rows:")
    display(duplicate_rows)
else:
    print("\nNo duplicate  rows found.")

<style>
h4 {
    margin-top: 0px;
    margin-bottom: 0px;
    font-weight: bold;
}
</style>

#### DEAL with DUPLICATE ROWS - Valeurs Foncieres

In [ ]:
# drop duplicate rows in df
#df = df.drop_duplicates()

# Save dataset in a new CSV file
df.to_csv("../data/processed/INTERIM_05_duplicates_ValeursFoncieres.csv", index=False)
print(f"✅ Dataset with duplicates handled and saved at ../data/processed/INTERIM_05_duplicates_ValeursFoncieres.csv")

<hr>

## 🕵️ REVISE & SAVE CLEAN DATA


<style>
h1 {
    text-align: center;
    color: black;
    font-weight: bold;
}
</style>

<style>
h2 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h3 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>

<style>
h4 {
    text-align: left;
    color: black;
    font-weight: bold;
}
</style>
<hr>

```text
- display head
- display info
- check unique values
- check data types
- check for nulls
- check for duplicates
```

In [ ]:
import pandas as pd

# --------------------------------------------
# Load dataset with duplicates handled
# --------------------------------------------
df = pd.read_csv("../data/processed/INTERIM_05_duplicates_ValeursFoncieres.csv")

# --------------------
# 🔲 DISPLAY - HEAD
# --------------------
# Display df DataFrame
print("Data Revision:")
display(df.head())
print("Dataset Shape:", df.shape[0], "rows and", df.shape[1], "columns\n")
print(100*"-")

# --------------------
# 🔲 DISPLAY - INFO
# --------------------
# display data types and missing values of each column in df
df.info()
print(100*"-")

# -----------------------------
# 🕵️ CHECK - UNIQUE VALUES
# -----------------------------
# Display data types & missing values of each column in df
print("Data Types & Missing Values of Each Column:")
display(df.info())

# Loop through each column and print unique values
for col in df.columns:
    unique_vals = df[col].unique()
    print(f"Column: {col}")
    print(f"Unique values ({len(unique_vals)}): {unique_vals}\n")
print(100*"-")

# -----------------------------
# 🕵️ CHECK - DATA TYPES
# -----------------------------
print(f"\nData types check:")
for col in df.columns:
    dtype = df[col].dtype
    unique_vals = df[col].nunique()
    print(f"  ➡️{col:<25} {str(dtype):<10} [{unique_vals}] unique values")
print(100*"-")

# -----------------------------
# 🕵️ CHECK - NULLS
# -----------------------------
# Loop through each column and print the number of nulls in each column
for col in df.columns:
    null_count = df[col].isnull().sum()
    print(f"[{col}]: [{null_count}] null values")
print(100*"-")


# -----------------------------
# 🕵️ CHECK - DUPLICATES
# -----------------------------
# Check for duplicate rows in the DataFrame
duplicate_rows = df[df.duplicated()]

# Print the number of duplicate rows
print(f"Number of duplicate rows: {duplicate_rows.shape[0]}")
print(100*"-")

# Display the duplicate rows if any exist
if not duplicate_rows.empty:
    print("\nDuplicate rows:")
    display(duplicate_rows)
else:
    print("\nNo duplicate  rows found.")
print(100*"-")

# --------------------------------------------
# Save CLEAN data in a new CSV file
# --------------------------------------------
df.to_csv("../data/processed/CLEAN_ValeursFoncieres_2020-S2_2025-S1.csv", index=False)
print(f"✅ Dataset cleaned and saved at ../data/processed/CLEAN_ValeursFoncieres_2020-S2_2025-S1.csv")